# Questão 4 — Dados Públicos

## Objetivo

Identificar produtos vendidos com prejuízo real, considerando:

- receita registrada em BRL no sistema de vendas
- custo unitário em USD no catálogo de importação
- conversão do custo para BRL com base na taxa de câmbio do dia da venda
- uso da última cotação disponível em dias sem câmbio (finais de semana e feriados)

## Premissas adotadas

- O custo em USD é unitário
- O custo total em BRL por transação considera a quantidade vendida
- O câmbio utilizado é a última cotação disponível menor ou igual à data da venda
- O prejuízo ocorre quando o custo total da transação é maior do que a receita da transação
- Impostos e frete foram desconsiderados, conforme enunciado

## Camadas utilizadas

- `03_silver.int_vendas_enriquecidas`: base transacional enriquecida com custo e câmbio
- `04_gold.fct_prejuizo_produto`: agregação final por produto

## Parte 1 — Cálculo e modelagem

Nesta etapa, são apresentados:

1. o cálculo do custo total em BRL por transação  
2. a identificação de transações com prejuízo  
3. a agregação por produto com:
   - receita total
   - prejuízo total
   - percentual de perda

In [0]:
%sql
SELECT
    sale_id,
    customer_id,
    product_id,
    quantity,
    sale_date,
    receita_transacao_brl,
    usd_price,
    exchange_rate_date,
    exchange_rate,
    custo_unitario_brl,
    custo_total_brl,
    prejuizo_brl,
    teve_prejuizo
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
ORDER BY prejuizo_brl DESC
LIMIT 20;

### Explicação da lógica transacional

A tabela `int_vendas_enriquecidas` foi construída com as seguintes regras:

- o custo histórico do produto foi selecionado pela data de vigência mais recente menor ou igual à data da venda
- a taxa de câmbio foi selecionada pela última data disponível menor ou igual à data da venda
- o custo total da transação foi calculado como:

`custo_total_brl = usd_price * exchange_rate * quantity`

- o prejuízo foi definido como:

`prejuizo_brl = custo_total_brl - receita_transacao_brl`, apenas quando o custo total superou a receita

In [0]:
%sql
SELECT
    id_produto,
    receita_total_brl,
    prejuizo_total_brl,
    percentual_perda
FROM lh_nautical.`04_gold`.fct_prejuizo_produto
ORDER BY prejuizo_total_brl DESC;

## Questão 4.1 — Código SQL

Abaixo está o SQL de agregação final por produto, contendo:

- receita total
- prejuízo total
- percentual de perda

In [0]:
%sql
SELECT
    product_id AS id_produto,
    SUM(receita_transacao_brl) AS receita_total_brl,
    SUM(prejuizo_brl) AS prejuizo_total_brl,
    CASE
        WHEN SUM(receita_transacao_brl) = 0 THEN 0
        ELSE SUM(prejuizo_brl) / SUM(receita_transacao_brl)
    END AS percentual_perda
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
GROUP BY product_id
ORDER BY prejuizo_total_brl DESC;

## Parte 2 — Análise visual

O gráfico abaixo apresenta o prejuízo total por produto, considerando apenas os produtos que registraram prejuízo no período analisado.

Observação: o produto 72 apresenta um valor muito superior aos demais, o que deve ser destacado na interpretação.

In [0]:
import matplotlib.pyplot as plt

# Leitura da tabela final
df_prejuizo = spark.table("lh_nautical.04_gold.fct_prejuizo_produto")

# Filtra apenas produtos com prejuízo
df_plot = (
    df_prejuizo
    .filter("prejuizo_total_brl > 0")
    .orderBy("prejuizo_total_brl", ascending=False)
    .toPandas()
)

# Criação do gráfico
plt.figure(figsize=(14, 7))
plt.bar(df_plot["id_produto"].astype(str), df_plot["prejuizo_total_brl"])

# Configurações do gráfico
plt.title("Prejuízo total por produto")
plt.xlabel("id_produto")
plt.ylabel("prejuízo_total_brl")
plt.xticks(rotation=90)
plt.tight_layout()

# Exibição
plt.show()

## Parte 3 — Análise objetiva

Nesta etapa, são respondidas objetivamente as perguntas do enunciado com base na tabela final agregada.

In [0]:
%sql
-- maior prejuízo absoluto

SELECT
    id_produto,
    receita_total_brl,
    prejuizo_total_brl,
    percentual_perda
FROM lh_nautical.`04_gold`.fct_prejuizo_produto
ORDER BY prejuizo_total_brl DESC
LIMIT 1;

In [0]:
%sql
-- maior percentual de perda

SELECT
    id_produto,
    receita_total_brl,
    prejuizo_total_brl,
    percentual_perda
FROM lh_nautical.`04_gold`.fct_prejuizo_produto
ORDER BY percentual_perda DESC
LIMIT 1;

## Respostas objetivas

### Qual produto concentra o maior prejuízo absoluto?
O produto **72** concentra o maior prejuízo absoluto no período analisado.

### O produto com maior prejuízo absoluto também é o que possui a maior porcentagem de perda?
**Sim.**

### Qual é o id_produto com maior porcentagem de perda financeira relativa?
O `id_produto` com maior porcentagem de perda financeira relativa é o **72**.

## Questão 4.2 — Validação

Com base na tabela agregada por produto, o `id_produto` que apresentou a maior porcentagem de perda financeira relativa no período analisado foi o **72**.

## Questão 4.3 — Interpretação

### Qual data de câmbio foi utilizada?
Foi utilizada a **última cotação disponível menor ou igual à data da venda**.

Isso significa que:
- em dias úteis com cotação, foi utilizada a cotação do próprio dia
- em finais de semana e feriados, foi utilizada a última cotação útil anterior

### Como o prejuízo foi definido?
O prejuízo foi definido quando o **custo total da transação em BRL** foi maior do que a **receita da transação em BRL**.

A fórmula utilizada foi:

`custo_total_brl = usd_price * exchange_rate * quantity`

e:

`prejuizo_brl = custo_total_brl - receita_transacao_brl`, quando o resultado foi positivo

### Alguma suposição relevante?
Sim. As principais suposições adotadas foram:

Apesar do produto 72 concentrar o maior prejuízo absoluto e relativo, o produto 83 também apresenta um nível crítico de perda, indicando possíveis problemas de precificação ou custo elevado, mesmo com menor volume financeiro.

o resultado mostra:
Produto 72 → prejuízo ≈ 39.8 milhões
Produto 83 → prejuízo ≈ 18.6 milhões

Após análise, podemos ver que 100% das transações dos produtos 72 e 83 ocorreram com prejuízo, evidenciando um problema estrutural de precificação. Isso indica que esses produtos estão sendo vendidos sistematicamente abaixo do custo, resultando em perdas financeiras recorrentes e significativas para a operação.

Observa-se na sequencia, que os produtos 74, 71 e 55 também apresentem percentuais de perda, embora muito menores em relação aos principais(72 e 83) . A análise detalhada revelou que a maioria de suas transações ocorre com prejuízo (74% a 99%).



Outros pontos:

- a ausência de cotação em finais de semana e feriados foi tratada com a última cotação útil anterior, O período de cambio buscado via api foi de : 01/12/2022 a 31/12/2024. O período necessário utilizado foi de 30/12/2022 a 31/12/2024. Considerei isso pois o dia 01/01/2023 foi um domingo.

- impostos e frete foram desconsiderados, conforme orientação do enunciado
- a base de câmbio foi expandida para incluir datas anteriores ao início das vendas, garantindo cobertura completa para o período analisado

### + Explorações

In [0]:
%sql
SELECT
    product_id,
    AVG(receita_transacao_brl / quantity) AS preco_medio,
    AVG(custo_unitario_brl) AS custo_medio,
    AVG((receita_transacao_brl / quantity) - custo_unitario_brl) AS margem_unitaria,
    AVG(
        ((receita_transacao_brl / quantity) - custo_unitario_brl)
        / (receita_transacao_brl / quantity)
    ) AS margem_percentual
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
WHERE product_id IN (72, 83)
GROUP BY product_id;

In [0]:
%sql
SELECT
    product_id,
    COUNT(*) AS total_transacoes,
    SUM(CASE WHEN teve_prejuizo THEN 1 ELSE 0 END) AS com_prejuizo,
    SUM(CASE WHEN teve_prejuizo THEN 1 ELSE 0 END) / COUNT(*) AS proporcao_prejuizo
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
WHERE product_id IN (72, 83)
GROUP BY product_id;

In [0]:
%sql
SELECT
    product_id,
    COUNT(*) AS total_transacoes,
    SUM(CASE WHEN teve_prejuizo THEN 1 ELSE 0 END) AS com_prejuizo,
    SUM(CASE WHEN teve_prejuizo THEN 1 ELSE 0 END) / COUNT(*) AS proporcao_prejuizo
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
WHERE product_id IN (72, 83)
GROUP BY product_id;

In [0]:
%sql
SELECT
    product_id,
    COUNT(*) AS total_transacoes,
    SUM(CASE WHEN teve_prejuizo THEN 1 ELSE 0 END) AS com_prejuizo,
    ROUND(
        SUM(CASE WHEN teve_prejuizo THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS proporcao_prejuizo
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
WHERE product_id IN (74, 71, 55)
GROUP BY product_id
ORDER BY proporcao_prejuizo DESC;

In [0]:
%sql
SELECT
    product_id,
    COUNT(*) AS total_transacoes,
    SUM(CASE WHEN teve_prejuizo THEN 1 ELSE 0 END) AS qtd_prejuizo,
    SUM(CASE WHEN NOT teve_prejuizo THEN 1 ELSE 0 END) AS qtd_lucro
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
WHERE product_id IN (74, 71, 55)
GROUP BY product_id;

In [0]:
%sql
SELECT
    product_id,
    AVG((receita_transacao_brl / quantity) - custo_unitario_brl) AS margem_unitaria_media
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
WHERE product_id IN (74, 71, 55)
GROUP BY product_id;

In [0]:
%sql
SELECT
    product_id,
    MIN(prejuizo_brl) AS min_prejuizo,
    MAX(prejuizo_brl) AS max_prejuizo,
    AVG(prejuizo_brl) AS avg_prejuizo
FROM lh_nautical.`03_silver`.int_vendas_enriquecidas
WHERE product_id IN (74, 71, 55)
GROUP BY product_id;

In [0]:
%sql
-- prejuízo potencial de precificação

SELECT
    product_id AS id_produto,

    -- Receita total
    SUM(receita_transacao_brl) AS receita_total_brl,

    -- Prejuízo real (já calculado)
    SUM(prejuizo_brl) AS prejuizo_real_brl,

    -- Gap de precificação (quanto vendeu abaixo do custo)
    SUM(
        (custo_unitario_brl - (receita_transacao_brl / quantity)) * quantity
    ) AS prejuizo_potencial_brl,

    -- Preço médio de venda
    AVG(receita_transacao_brl / quantity) AS preco_medio_venda,

    -- Custo médio
    AVG(custo_unitario_brl) AS custo_medio,

    -- Margem unitária média
    AVG(
        (receita_transacao_brl / quantity) - custo_unitario_brl
    ) AS margem_unitaria_media,

    -- Margem percentual média
    AVG(
        ((receita_transacao_brl / quantity) - custo_unitario_brl)
        / (receita_transacao_brl / quantity)
    ) AS margem_percentual_media

FROM lh_nautical.`03_silver`.int_vendas_enriquecidas

WHERE product_id IN (72, 83)

GROUP BY product_id
ORDER BY prejuizo_real_brl DESC;

In [0]:
%sql
SELECT
    product_id,
    COUNT(*) AS qtd
FROM lh_nautical.02_bronze.produtos_bronze
GROUP BY product_id
HAVING COUNT(*) > 1
ORDER BY qtd DESC, product_id;

In [0]:
%sql
select
    sum(receita_transacao_brl) as receita_total,
    sum(custo_total_brl) as custo_total,
    sum(prejuizo_brl) as prejuizo_total
from lh_nautical.04_gold.fct_vendas;

In [0]:
%sql
select
    min(data_dia) as data_minima,
    max(data_dia) as data_maxima,
    count(*) as total_dias
from lh_nautical.04_gold.dim_data;